# Tool Calling Test Notebook

This notebook tests the ai-jup-style tool calling features:
- `$`variable`` - Reference kernel variables in prompts
- `&`function`` - Expose Python functions as AI-callable tools
- Built-in file tools (view, rg, create, str_replace, insert)
- Tool loop for multi-step operations

In [ ]:
# Setup test variables
test_list = [1, 2, 3, 4, 5]
test_dict = {'name': 'Alice', 'age': 30}
test_string = "Hello, World!"

# Try to create a DataFrame if pandas is available
test_df = None
try:
    import pandas as pd
    test_df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6], 'c': ['x', 'y', 'z']})
    print(f'test_df:\n{test_df}')
except ImportError:
    print('pandas not available, test_df will be None')

print(f'test_list: {test_list}')
print(f'test_dict: {test_dict}')
print(f'test_string: {test_string}')

## Test 1: Variable Substitution

The prompt below references `$`test_list`` and `$`test_dict``.
The values should be substituted before sending to the LLM.

What is the sum of $`test_list`? Also describe the person in $`test_dict`.

---

*Run this prompt cell to test variable substitution*

In [ ]:
# Define test functions for tool calling

def greet(name: str) -> str:
    """Greet a person by name.
    
    Args:
        name: The name of the person to greet
    
    Returns:
        A greeting message
    """
    return f'Hello, {name}! Welcome to Dialeng.'


def calculate_stats(numbers: list) -> dict:
    """Calculate basic statistics for a list of numbers.
    
    Args:
        numbers: List of numeric values
    
    Returns:
        Dictionary with mean, min, max, and sum
    """
    if not numbers:
        return {'error': 'Empty list provided'}
    return {
        'mean': sum(numbers) / len(numbers),
        'min': min(numbers),
        'max': max(numbers),
        'sum': sum(numbers),
        'count': len(numbers)
    }


def multiply(a: int, b: int) -> int:
    """Multiply two numbers together.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Product of a and b
    """
    return a * b


def format_currency(amount: float, currency: str = 'USD') -> str:
    """Format a number as currency.
    
    Args:
        amount: The monetary amount
        currency: Currency code (default: USD)
    
    Returns:
        Formatted currency string
    """
    symbols = {'USD': '$', 'EUR': '\u20ac', 'GBP': '\u00a3', 'JPY': '\u00a5'}
    symbol = symbols.get(currency, currency)
    return f'{symbol}{amount:,.2f}'


print('Functions defined: greet, calculate_stats, multiply, format_currency')

## Test 2: Single Tool Call

The prompt below uses `&`greet`` to expose the greet function as a tool.

Use &`greet` to say hello to Bob.

---

*Run this prompt cell to test single tool calling*

## Test 3: Tool with Variable

Combines `$`variable`` substitution with `&`function`` tool calling.

Use &`calculate_stats` to analyze $`test_list` and tell me the results.

---

*Run this prompt cell to test tool + variable combination*

## Test 4: Multi-Tool Workflow

Uses multiple tools in sequence (tool loop).

First use &`multiply` to calculate 7 * 8, then use &`format_currency` to format that result as USD.

---

*Run this prompt cell to test multi-tool workflow*

## Test 5: Built-in Tool - view

Uses the built-in `view` tool to read a file (no `&` prefix needed).

Use the view tool to show me the first 20 lines of app.py and tell me what the file is about.

---

*Run this prompt cell to test built-in view tool*

## Test 6: Built-in Tool - rg (ripgrep)

Uses the built-in `rg` tool to search for patterns.

Use the rg tool to find all function definitions ("def ") in the services directory and list the top 10.

---

*Run this prompt cell to test built-in rg tool*

## Test 7: Note Cell Tool Declaration

This note cell declares available tools that the next prompt can use:

### Available Analysis Tools
- &`greet` - Say hello to someone
- &`calculate_stats` - Calculate statistics
- &`multiply` - Multiply numbers
- &`format_currency` - Format as currency

Current test values: $`test_list` has 5 items.

Using the tools declared above, calculate stats on some sample numbers [10, 20, 30, 40] and format the sum as EUR currency.

---

*Run this prompt cell to test note cell tool declarations*

## Test 8: Error Handling - Invalid Variable

Tests what happens when referencing a non-existent variable.

What is $`undefined_variable`?

---

*Run this prompt cell to test error handling for undefined variables*

## Test 9: Error Handling - Invalid Function

Tests what happens when referencing a non-existent function.

Use &`nonexistent_function` to do something.

---

*Run this prompt cell to test error handling for undefined functions*

## Expected Results Summary

| Test | Expected Behavior |
|------|-------------------|
| Test 1 | Variable values substituted in prompt |
| Test 2 | AI calls greet() and shows result |
| Test 3 | AI calls calculate_stats with test_list data |
| Test 4 | AI calls multiply, then format_currency in sequence |
| Test 5 | Built-in view tool shows app.py content |
| Test 6 | Built-in rg tool finds function definitions |
| Test 7 | Tools declared in note cell are available |
| Test 8 | Shows error for undefined variable |
| Test 9 | Shows error for undefined function |